# Generate Car Point Cloud sampples for classification
(x, y, z, v),
corrdinates are centralized

## Environment Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
MYDATASET_RADAR_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyDataset_rsu1/radar"
MYDATASET_CAR_BOX_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyDataset_rsu1/car_box"
MYMODELNET_CAR_PATH = "/content/drive/MyDrive/THESIS_dataset/mmw/MyModelNet_cls/car"

In [4]:
import sys
sys.path.append('/content/drive/MyDrive/THESIS/code')

import numpy as np
import os
from local_visualize.utils.car_box2label import is_car

## Extract car point clouds as samples

In [15]:
def generate_car_samples(index, counter_car):
  radar_txt_path = os.path.join(MYDATASET_RADAR_PATH, index + ".txt")
  radar_points_labels = np.loadtxt(radar_txt_path, delimiter=',')
  xyz_ls = radar_points_labels[:,0:3]
  xyzv_ls = radar_points_labels[:,0:4]

  #%%
  # load car box file
  car_corners_list_path = os.path.join(MYDATASET_CAR_BOX_PATH, index + ".npy")
  car_corners_list = np.load(car_corners_list_path)
  # delete boxes that are out of sight
  car_box_centers_ls = car_corners_list.mean(axis=1)
  mask = (car_box_centers_ls[:,0] < 0) & (car_box_centers_ls[:,1] < 0)
  car_corners_list = car_corners_list[mask]
  num_car = car_corners_list.shape[0]

  #%%
  for i_car in range(num_car):
      ps = []
      corners = car_corners_list[i_car]
      corners = np.array([corners])

      for i, point in enumerate(xyz_ls):
          if is_car(point, corners):
              ps.append(xyzv_ls[i])

      if ps != []:
        ps = np.array(ps)
        centroid = np.mean(ps[:, :3], axis=0)
        ps[:, 0:3] = ps[:, 0:3] - centroid

        car_save_pth = os.path.join(MYMODELNET_CAR_PATH, f"car_{counter_car:05d}.txt")
        np.savetxt(car_save_pth, ps, fmt='%.6f', delimiter=',')
        counter_car += 1

  return counter_car

In [16]:
index_range = ["016653", "017853"] # 017853
index = index_range[0]

counter_car = 1
while index != index_range[1]:
    counter_car = generate_car_samples(index, counter_car)

    index = f"{int(index) + 1:06d}"
    if counter_car % 100 == 0:
        print(index)

016941
016966
017047
017148
017230
017324
017369
017419
017469
017519
017569
017619
017801
